# Region Aware Blind Face Restoration — DifFace Gradio Demo

**Before running:** `Runtime` → `Change runtime type` → **GPU (T4)**.

Cells:
1. **Setup** — clone DifFace, install deps.
2. **Download fine-tuned weights from a public Drive link**.
3. **Launch Gradio** — public `share=True` link with three tabs:
   - Restoration on an aligned 512×512 face
   - Restoration on a whole image
   - **Generate Synthetic Sample** — apply the GFPGAN-style degradation to your HQ image and download the result

## 1. Setup — clone DifFace, install deps

In [1]:
import os, shutil
from pathlib import Path

CODE_DIR = Path('/content/DifFace')
if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)
!git clone https://github.com/zsyOAOA/DifFace.git $CODE_DIR
%cd $CODE_DIR

!pip install -q -r requirements.txt
!pip install -q gradio gdown

print('\nSetup done.')

fatal: could not create leading directories of '/content/DifFace': Read-only file system
[Errno 2] No such file or directory: '/content/DifFace'
/Users/ibtesam/Downloads


/Users/ibtesam/Library/Python/3.9/lib/python/site-packages/IPython/core/magics/osm.py:393: UserWarning: using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})


ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11
ERROR: Could not find a version that satisfies the requirement nvidia-cublas-cu11==11.11.3.6 (from versions: 0.0.1.dev5, 0.0.1)

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
ERROR: No matching distribution found for nvidia-cublas-cu11==11.11.3.6

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

Setup done.


## 2. Download fine-tuned weights from a public Drive link

In Drive, right-click the `.pth` file → **Share** → set access to **Anyone with the link → Viewer**.

Then paste the share URL (or just the file ID) below and run the cell.

In [2]:
import re, os
from pathlib import Path


DRIVE_FILE = ''

m = re.search(r'/d/([a-zA-Z0-9_-]+)|id=([a-zA-Z0-9_-]+)', DRIVE_FILE)
file_id = (m.group(1) or m.group(2)) if m else DRIVE_FILE.strip()

dst = Path('/content/finetuned_difface.pth')
if not file_id or 'PASTE' in file_id:
    print('Edit DRIVE_FILE first, then re-run this cell.')
else:
    print(f'Downloading file_id = {file_id}')
    !gdown --id $file_id -O $dst
    if dst.exists():
        size_mb = os.path.getsize(dst) / 1e6
        print(f'\nDownloaded -> {dst} ({size_mb:.1f} MB)')
    else:
        print('\nDownload failed. Confirm the file is shared with "Anyone with the link".')

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1tcC-BkT9ED2d0uPgHfCEMszQpfwX_ALV
From (redirected): https://drive.google.com/uc?id=1tcC-BkT9ED2d0uPgHfCEMszQpfwX_ALV&confirm=t&uuid=6004f7ad-d683-4862-93e0-8e4203f4d0aa
To: /content/finetuned_difface.pth
100% 639M/639M [00:03<00:00, 203MB/s]

Downloaded -> /content/finetuned_difface.pth (639.4 MB)


## 3. Launch Gradio with model selector and synthetic-sample generator

First inference call downloads the pretrained DifFace weights (~1 min, one time). The public URL is printed at the end.

In [3]:
import os, sys, shutil, subprocess, glob, io, math, random, time
from pathlib import Path
import numpy as np
import cv2
from PIL import Image
import gradio as gr
import torch

REPO_DIR = Path('/content/DifFace')
WEIGHTS_DIR = REPO_DIR / 'weights'
PRETRAINED_BACKUP = Path('/content/_pretrained_backup.pth')
FINETUNED_SRC = Path('/content/finetuned_difface.pth')
USE_FP16 = torch.cuda.is_available()
if not USE_FP16:
    print('WARNING: No GPU detected. Switch Runtime to GPU (T4).')

TMP_IN = Path('/content/_in')
TMP_OUT_ALIGNED = Path('/content/_out_aligned')
TMP_OUT_WHOLE = Path('/content/_out_whole')
SYN_OUT_DIR = Path('/content/_synthetic')
SYN_OUT_DIR.mkdir(exist_ok=True)

def _reset(p: Path) -> Path:
    if p.exists():
        shutil.rmtree(p)
    p.mkdir(parents=True, exist_ok=True)
    return p

def _write_input(image_rgb) -> Path:
    d = _reset(TMP_IN)
    bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
    cv2.imwrite(str(d / 'input.png'), bgr)
    return d


# ---------------- Model swap (Pretrained vs Fine-tuned) ----------------
def _find_diffusion_weight():
    if not WEIGHTS_DIR.exists():
        return None
    candidates = sorted(WEIGHTS_DIR.rglob('*diffusion*.pt*'))
    if not candidates:
        candidates = sorted(WEIGHTS_DIR.rglob('*.pt')) + sorted(WEIGHTS_DIR.rglob('*.pth'))
    if not candidates:
        return None
    return max(candidates, key=lambda p: p.stat().st_size)

def _set_active_weights(use_finetuned: bool):
    target = _find_diffusion_weight()
    if target is None:
        return
    if not PRETRAINED_BACKUP.exists():
        shutil.copy(target, PRETRAINED_BACKUP)
    if use_finetuned:
        if not FINETUNED_SRC.exists():
            raise gr.Error('Fine-tuned weights not loaded. Run cell 2 first.')
        shutil.copy(FINETUNED_SRC, target)
        print(f'Active weights: FINE-TUNED -> {target.name}')
    else:
        shutil.copy(PRETRAINED_BACKUP, target)
        print(f'Active weights: PRETRAINED -> {target.name}')


# ---------------- DifFace inference ----------------
def _run(in_dir: Path, out_dir: Path, task: str, aligned: bool, eta: float):
    cmd = [
        sys.executable,
        str(REPO_DIR / 'inference_difface.py'),
        '-i', str(in_dir),
        '-o', str(out_dir),
        '--task', task,
        '--eta', str(eta),
    ]
    if aligned and task == 'restoration':
        cmd.append('--aligned')
    if USE_FP16:
        cmd.append('--use_fp16')
    if task == 'restoration' and not aligned:
        cmd += ['--bs', '1']
    print('RUN:', ' '.join(cmd))
    proc = subprocess.run(cmd, cwd=str(REPO_DIR), capture_output=True, text=True)
    if proc.returncode != 0:
        tail = (proc.stderr or proc.stdout)[-1200:]
        raise gr.Error(f'DifFace failed:\n{tail}')

def _read_first(out_dir: Path, subs):
    for sub in subs:
        d = out_dir / sub
        if not d.exists():
            continue
        files = sorted(d.glob('*.png')) + sorted(d.glob('*.jpg'))
        if files:
            bgr = cv2.imread(str(files[0]))
            return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    raise gr.Error(f'No output image produced in {out_dir}')

def restore_aligned(image, model_choice, eta):
    if image is None:
        return None
    in_dir = _write_input(image)
    out_dir = _reset(TMP_OUT_ALIGNED)
    if WEIGHTS_DIR.exists() and any(WEIGHTS_DIR.iterdir()):
        _set_active_weights(model_choice == 'Fine-tuned (Saudi)')
    _run(in_dir, out_dir, 'restoration', aligned=True, eta=float(eta))
    _set_active_weights(model_choice == 'Fine-tuned (Saudi)')
    return _read_first(out_dir, ['restored_faces'])

def restore_whole(image, model_choice, eta):
    if image is None:
        return None
    in_dir = _write_input(image)
    out_dir = _reset(TMP_OUT_WHOLE)
    if WEIGHTS_DIR.exists() and any(WEIGHTS_DIR.iterdir()):
        _set_active_weights(model_choice == 'Fine-tuned (Saudi)')
    _run(in_dir, out_dir, 'restoration', aligned=False, eta=float(eta))
    _set_active_weights(model_choice == 'Fine-tuned (Saudi)')
    return _read_first(out_dir, ['restored_image', 'restored_faces'])


# ---------------- GFPGAN-style synthetic degradation ----------------
def gfpgan_degrade(image_rgb, blur_sigma, scale, noise_sigma, jpeg_quality, seed=None):
    if image_rgb is None:
        return None, None
    if seed is not None and seed >= 0:
        np.random.seed(int(seed))
        random.seed(int(seed))

    h, w = image_rgb.shape[:2]
    bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR).astype(np.float32) / 255.0

    k = 41
    blurred = cv2.GaussianBlur(bgr, (k, k), max(float(blur_sigma), 1e-3))

    s = max(float(scale), 1.0)
    new_w = max(1, int(w / s))
    new_h = max(1, int(h / s))
    down = cv2.resize(blurred, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    noise_std = max(float(noise_sigma), 0.0) / 255.0
    if noise_std > 0:
        noise = np.random.normal(0, noise_std, down.shape).astype(np.float32)
        down = np.clip(down + noise, 0.0, 1.0)

    down_uint8 = (down * 255.0).clip(0, 255).astype(np.uint8)
    pil = Image.fromarray(cv2.cvtColor(down_uint8, cv2.COLOR_BGR2RGB))
    buf = io.BytesIO()
    pil.save(buf, format='JPEG', quality=int(jpeg_quality))
    buf.seek(0)
    compressed_rgb = np.array(Image.open(buf))
    compressed_bgr = cv2.cvtColor(compressed_rgb, cv2.COLOR_RGB2BGR)

    final_bgr = cv2.resize(compressed_bgr, (w, h), interpolation=cv2.INTER_LINEAR)
    final_rgb = cv2.cvtColor(final_bgr, cv2.COLOR_BGR2RGB)

    fname = SYN_OUT_DIR / f'degraded_{int(time.time())}.png'
    cv2.imwrite(str(fname), final_bgr)
    return final_rgb, str(fname)


# ---------------- UI ----------------
TITLE = 'Region Aware Blind Face Restoration — DifFace'
DESCRIPTION = '''
Demo of the project's DifFace baseline (ICS619 — MX-FR-T251).

- **Pretrained** — original DifFace weights.
- **Fine-tuned (Saudi)** — downloaded from Drive in cell 2 of this notebook.
- **Generate Synthetic Sample** — apply the GFPGAN-style degradation pipeline to your own HQ image and download the result.
'''

with gr.Blocks(title=TITLE) as demo:
    gr.Markdown(f'# {TITLE}')
    gr.Markdown(DESCRIPTION)
    with gr.Tabs():
        with gr.Tab('Restoration (aligned face)'):
            with gr.Row():
                with gr.Column():
                    in1 = gr.Image(type='numpy', label='Aligned 512×512 face')
                    model1 = gr.Radio(['Pretrained', 'Fine-tuned (Saudi)'], value='Pretrained', label='Model')
                    eta1 = gr.Slider(0.0, 1.0, value=1.0, step=0.05, label='Eta')
                    btn1 = gr.Button('Restore', variant='primary')
                with gr.Column():
                    out1 = gr.Image(type='numpy', label='Restored')
            btn1.click(fn=restore_aligned, inputs=[in1, model1, eta1], outputs=out1)

        with gr.Tab('Restoration (whole image)'):
            with gr.Row():
                with gr.Column():
                    in2 = gr.Image(type='numpy', label='Input photo')
                    model2 = gr.Radio(['Pretrained', 'Fine-tuned (Saudi)'], value='Pretrained', label='Model')
                    eta2 = gr.Slider(0.0, 1.0, value=1.0, step=0.05, label='Eta')
                    btn2 = gr.Button('Restore', variant='primary')
                with gr.Column():
                    out2 = gr.Image(type='numpy', label='Restored')
            btn2.click(fn=restore_whole, inputs=[in2, model2, eta2], outputs=out2)

        with gr.Tab('Generate Synthetic Sample'):
            gr.Markdown('Upload a high-quality face and apply the **GFPGAN synthetic degradation pipeline** (blur → downsample → noise → JPEG → resize back). Sliders control the strength of each stage. Download the result with the link below the output.')
            with gr.Row():
                with gr.Column():
                    in3 = gr.Image(type='numpy', label='High-quality input')
                    blur = gr.Slider(0.1, 10.0, value=4.0, step=0.1, label='Blur sigma (Gaussian)')
                    scale = gr.Slider(1.0, 8.0, value=4.0, step=0.5, label='Downsample scale factor')
                    noise = gr.Slider(0.0, 20.0, value=10.0, step=1.0, label='Gaussian noise sigma')
                    jpeg = gr.Slider(40, 100, value=70, step=5, label='JPEG quality (lower = more artefacts)')
                    seed = gr.Number(value=-1, label='Seed (-1 for random)', precision=0)
                    btn3 = gr.Button('Generate degraded sample', variant='primary')
                with gr.Column():
                    out3 = gr.Image(type='numpy', label='Synthetic low-quality output')
                    file3 = gr.File(label='Download degraded image')
            btn3.click(fn=gfpgan_degrade, inputs=[in3, blur, scale, noise, jpeg, seed], outputs=[out3, file3])

demo.queue().launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a7834b452efe037fe5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
